# Phase 5: LLM Team Explainer

Generates natural-language explanations for the optimizer's fantasy XI using the Gemini API.

**Run cells top to bottom.** Cell 1 must run first (sets the OpenMP flag before torch/lightgbm import, which prevents the kernel crash on macOS).

In [ ]:
# CELL 1 — MUST RUN FIRST, before any torch/lightgbm import
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import pickle
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn

import pandas as pd
import numpy as np
import json
import time

print("imports ok")

In [ ]:
# CELL 2 — load data and config
df = pd.read_csv("data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

credits = pd.read_csv("data/processed/player_credits.csv")

with open("models/ensemble_config.json") as f:
    ensemble_config = json.load(f)

sequence_features = ensemble_config["sequence_features"]
context_features = ensemble_config["context_features"]
SEQ_LEN = ensemble_config["seq_len"]

print("data ok:", df.shape, credits.shape)
print("seq/context/len:", len(sequence_features), len(context_features), SEQ_LEN)

In [ ]:
# CELL 3 — load LightGBM
with open("models/lgbm_final.pkl", "rb") as f:
    lgbm = pickle.load(f)

lgbm_features = lgbm.feature_name_
print("lgbm ok,", len(lgbm_features), "features")

In [ ]:
# CELL 4 — load LSTM
class CricketLSTM(nn.Module):
    def __init__(self, seq_features, context_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=seq_features, hidden_size=hidden_size,
                             num_layers=2, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_size + context_features, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, seq, context):
        lstm_out, (hidden, cell) = self.lstm(seq)
        last_output = lstm_out[:, -1, :]
        combined = torch.cat([last_output, context], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(1)

device = torch.device("mps")
model = CricketLSTM(seq_features=len(sequence_features),
                     context_features=len(context_features),
                     hidden_size=32)
model.load_state_dict(torch.load("models/lstm_final.pt", map_location=device))
model = model.to(device)
model.eval()

print("lstm ok")

In [ ]:
# CELL 5 — optimizer functions
import pulp

def select_fantasy_team(match_pool, budget=100, min_wk=1, max_wk=4,
                          min_batters=3, max_batters=6,
                          min_bowlers=3, max_bowlers=6,
                          min_allrounders=1, max_allrounders=4,
                          max_per_team=7):

    players = match_pool["player"].tolist()
    points = dict(zip(match_pool["player"], match_pool["ensemble_pred"]))
    credits_map = dict(zip(match_pool["player"], match_pool["credit_value"]))
    roles = dict(zip(match_pool["player"], match_pool["role"]))
    teams = dict(zip(match_pool["player"], match_pool["team"]))

    prob = pulp.LpProblem("Fantasy_Team_Selection", pulp.LpMaximize)
    player_vars = {p: pulp.LpVariable(f"select_{p}", cat="Binary") for p in players}

    prob += pulp.lpSum([points[p] * player_vars[p] for p in players])
    prob += pulp.lpSum([player_vars[p] for p in players]) == 11
    prob += pulp.lpSum([credits_map[p] * player_vars[p] for p in players]) <= budget

    wk_players = [p for p in players if roles[p] == "wicketkeeper"]
    batter_players = [p for p in players if roles[p] == "batter"]
    bowler_players = [p for p in players if roles[p] == "bowler"]
    allrounder_players = [p for p in players if roles[p] == "allrounder"]

    prob += pulp.lpSum([player_vars[p] for p in wk_players]) >= min_wk
    prob += pulp.lpSum([player_vars[p] for p in wk_players]) <= max_wk
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) >= min_batters
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) <= max_batters
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) >= min_bowlers
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) <= max_bowlers
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) >= min_allrounders
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) <= max_allrounders

    for team_name in match_pool["team"].unique():
        team_players = [p for p in players if teams[p] == team_name]
        prob += pulp.lpSum([player_vars[p] for p in team_players]) <= max_per_team

    prob.solve(pulp.PULP_CBC_CMD(msg=0))

    status = pulp.LpStatus[prob.status]
    if status != "Optimal":
        print(f"Warning: solver status is {status}")
        return None

    selected = [p for p in players if player_vars[p].value() == 1]
    result = (match_pool[match_pool["player"].isin(selected)]
              .sort_values("ensemble_pred", ascending=False)
              .reset_index(drop=True))

    result["captain"] = False
    result["vice_captain"] = False
    result.loc[0, "captain"] = True
    result.loc[1, "vice_captain"] = True

    result["final_points"] = result["ensemble_pred"]
    result.loc[result["captain"], "final_points"] *= 2.0
    result.loc[result["vice_captain"], "final_points"] *= 1.5

    return result


def predict_and_select_team(match_df):
    match_df = match_df.copy()

    match_df["lgbm_pred"] = lgbm.predict(match_df[lgbm_features])

    lstm_preds_list = []
    for _, row in match_df.iterrows():
        history = df[(df["player"] == row["player"]) & (df["date"] < row["date"])].sort_values("date")
        hist_data = history[sequence_features].values
        if len(hist_data) < SEQ_LEN:
            pad_len = SEQ_LEN - len(hist_data)
            padding = np.zeros((pad_len, len(sequence_features)))
            seq = np.vstack([padding, hist_data])
        else:
            seq = hist_data[-SEQ_LEN:]

        context = row[context_features].values.astype(float)
        seq_t = torch.FloatTensor(seq).unsqueeze(0).to(device)
        context_t = torch.FloatTensor(context).unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(seq_t, context_t).cpu().numpy()[0]
        lstm_preds_list.append(pred)

    match_df["lstm_pred"] = lstm_preds_list
    match_df["ensemble_pred"] = 0.5 * match_df["lgbm_pred"] + 0.5 * match_df["lstm_pred"]

    match_pool = match_df.drop(columns=["role", "role_encoded"], errors="ignore").merge(
        credits[["player", "role", "credit_value"]], on="player", how="left"
    )

    team = select_fantasy_team(match_pool)
    return team, match_pool

print("optimizer functions defined")

In [ ]:
# CELL 6 — generate a team for a test match
test_2025 = df[df["date"].dt.year >= 2025]
sample_match_id = test_2025["match_id"].iloc[0]
sample_match = df[df["match_id"] == sample_match_id].copy()

team, match_pool = predict_and_select_team(sample_match)

print(team[["player", "team", "role", "credit_value", "ensemble_pred",
             "captain", "vice_captain", "final_points"]])
print("\nCredits used:", team["credit_value"].sum())
print("Total predicted points:", team["final_points"].sum())

In [ ]:
# CELL 7 — Gemini client with retry/backoff
from dotenv import load_dotenv
load_dotenv()

from google import genai
client = genai.Client()

def call_gemini(prompt, model_name="gemini-flash-latest", max_retries=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model_name, contents=prompt)
            return response.text
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"Attempt {attempt+1} failed, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

print(call_gemini("Reply with exactly: connection ok"))

In [ ]:
# CELL 8 — build the explanation prompt
def build_explanation_prompt(team_df, team1, team2, venue):
    lines = []
    for _, row in team_df.iterrows():
        tag = ""
        if row["captain"]:
            tag = " (CAPTAIN, 2x points)"
        elif row["vice_captain"]:
            tag = " (VICE-CAPTAIN, 1.5x points)"
        lines.append(
            f"- {row['player']} ({row['team']}, {row['role']}) "
            f"- predicted {row['ensemble_pred']:.1f} pts, {row['credit_value']} credits{tag}"
        )

    team_list_str = "\n".join(lines)
    total_credits = team_df["credit_value"].sum()
    total_points = team_df["final_points"].sum()

    prompt = f"""You are a cricket fantasy analyst. Explain this fantasy XI selection for {team1} vs {team2} at {venue}.

Selected team ({total_credits:.1f}/100 credits used, {total_points:.1f} total projected points including captain multipliers):
{team_list_str}

Write a clear 150-200 word explanation covering:
1. Why the captain and vice-captain were chosen
2. The team's batting/bowling balance
3. One or two standout picks worth highlighting
4. A brief, honest note that these are model-based projections, not guarantees

Write in plain paragraphs. Do not use markdown, bullet points, or headers."""

    return prompt


prompt = build_explanation_prompt(
    team,
    sample_match["team"].iloc[0],
    sample_match["opposition"].iloc[0],
    sample_match["venue"].iloc[0]
)
print(prompt)

In [ ]:
# CELL 9 — generate the explanation
explanation = call_gemini(prompt)
print(explanation)